# Assignment 4: Transformer Representation Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/09/Assignment_04_Transformer_Representation_Analysis.ipynb)

**Course:** Neural Architectures and Representation Learning  
**Related notebook:** `Week_09_Transformers_Contextual_Representations.ipynb`


## Task

You will train and analyze a tiny transformer on an ambiguous-word task.

The goal is not to build a large language model. The goal is to show evidence that you understand how self-attention builds contextual representations.

You will submit:

1. A baseline tiny-transformer run.
2. Two controlled model experiments.
3. Attention heatmaps for contrasting examples.
4. Contextual representation analysis for `bank`.
5. Written interpretation of what the model captures and misses.


## Grading rubric

| Criterion | Points |
|----------|--------|
| Correctly runs the baseline transformer | 15 |
| Performs controlled transformer experiments | 20 |
| Uses attention heatmaps clearly | 20 |
| Analyzes contextual representations | 20 |
| Explains limitations and ambiguous examples | 15 |
| Clear, reproducible notebook | 10 |

Total: 100 points.


---

## Environment

This notebook uses `torch`, `numpy`, `matplotlib`, and `scikit-learn`. CPU is enough.

Do not change the fixed dataset cell unless an optional extension explicitly asks you to add examples.

## Colab pointers

Run cells in order. The trained models are reused by later visualization cells.

Main objects:

- `TinyTransformerClassifier.token_embedding`: static token vectors
- `TinyTransformerClassifier.pos_embedding`: position vectors
- `TinyTransformerClassifier.block.attn`: self-attention
- `TinyTransformerClassifier.classifier`: prediction from contextual `[CLS]`


## Optional references

These are optional references, not required dependencies for the assignment.

| Topic | Resource | Why it helps |
|---|---|---|
| Self-attention intuition | [3Blue1Brown: Attention in transformers, visually explained](https://www.youtube.com/watch?v=eMlx5fFNoYc) | Visual explanation of attention as moving information between token vectors. |
| Transformer walkthrough | [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) | Interactive view of tokenization, embeddings, attention, MLP blocks, and output probabilities. |
| Architecture overview | [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) | Visual recap of the full transformer block. |
| Attention inspection | [BertViz](https://github.com/jessevig/bertviz) | Optional tool for visualizing attention heads in pretrained transformer models. |
| Interpretation caveat | [Attention is not Explanation](https://arxiv.org/abs/1902.10186) | Reminder that attention maps are useful evidence, not guaranteed explanations. |

In [ ]:
import math
import random
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.decomposition import PCA

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(9)


---

## Fixed ambiguous-word dataset

The token `bank` appears in both classes. Context words decide the label.


In [ ]:
finance_sentences = [
    "bank approves loan",
    "bank manages savings",
    "bank account earns interest",
    "deposit money at bank",
    "cash teller works at bank",
    "credit card from bank",
    "bank invests client money",
    "loan officer calls from bank",
    "bank transfers cash today",
    "customer opens bank account",
    "bank charges account fee",
    "savings grow inside bank",
]

river_sentences = [
    "bank has mud near river",
    "river water touches bank",
    "fish swim by river bank",
    "trees grow on bank near water",
    "boat rests beside bank",
    "stream curves around bank",
    "bank erodes after rain",
    "ducks walk along river bank",
    "muddy bank borders river",
    "grass covers the river bank",
    "water rises over bank",
    "stone path follows bank",
]

labels = ["finance", "river"]
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

all_examples = [(s, label_to_idx["finance"]) for s in finance_sentences]
all_examples += [(s, label_to_idx["river"]) for s in river_sentences]
random.shuffle(all_examples)

TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

counter = Counter()
for text, _ in all_examples:
    counter.update(tokenize(text))

special_tokens = ["<PAD>", "<UNK>", "[CLS]"]
word_to_idx = {tok: i for i, tok in enumerate(special_tokens)}
for word in sorted(counter):
    word_to_idx[word] = len(word_to_idx)
idx_to_word = {i: word for word, i in word_to_idx.items()}
pad_idx = word_to_idx["<PAD>"]
cls_idx = word_to_idx["[CLS]"]

print("examples:", len(all_examples))
print("vocabulary size:", len(word_to_idx))
print("labels:", labels)
print("sample tokens:", tokenize(all_examples[0][0]))


In [ ]:
def encode_sentence(text):
    ids = [cls_idx]
    ids += [word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in tokenize(text)]
    return torch.tensor(ids, dtype=torch.long)


def collate_sentences(batch):
    encoded = [encode_sentence(text) for text, _ in batch]
    lengths = torch.tensor([len(x) for x in encoded], dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.full((len(batch), max_len), pad_idx, dtype=torch.long)
    for i, item in enumerate(encoded):
        x[i, : len(item)] = item
    return x, lengths, y

train_loader = DataLoader(all_examples, batch_size=8, shuffle=True, collate_fn=collate_sentences)

class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=2, ff_dim=64, dropout=0.0):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None, return_attention=False):
        attn_out, attn_weights = self.attn(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        if return_attention:
            return x, attn_weights
        return x

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model=32, num_heads=2, ff_dim=64, max_len=16):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.block = TinyTransformerBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, return_attention=False, return_representations=False):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        static = self.token_embedding(x)
        h0 = static + self.pos_embedding(positions)
        key_padding_mask = x == pad_idx
        h, attn = self.block(h0, key_padding_mask=key_padding_mask, return_attention=True)
        logits = self.classifier(h[:, 0])
        if return_representations:
            return logits, attn, static, h0, h
        if return_attention:
            return logits, attn
        return logits


def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_transformer(d_model=32, num_heads=2, ff_dim=64, epochs=120, lr=0.01, seed=9):
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")
    set_seed(seed)
    model = TinyTransformerClassifier(
        vocab_size=len(word_to_idx),
        num_classes=len(labels),
        d_model=d_model,
        num_heads=num_heads,
        ff_dim=ff_dim,
        max_len=16,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "accuracy": []}

    for epoch in range(epochs):
        model.train()
        losses, accs = [], []
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            accs.append(accuracy_from_logits(logits, y))
        history["loss"].append(float(np.mean(losses)))
        history["accuracy"].append(float(np.mean(accs)))
    return model, history


def plot_training_history(history, title="Tiny transformer training"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(history["loss"])
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[1].plot(history["accuracy"])
    axes[1].set_title("Training accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1.05)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

transformer_model, transformer_history = train_transformer()
plot_training_history(transformer_history)
print("final training accuracy:", round(transformer_history["accuracy"][-1], 3))


In [ ]:
@torch.no_grad()
def model_details(model, sentence):
    model.eval()
    x, lengths, _ = collate_sentences([(sentence, 0)])
    x = x.to(device)
    logits, attn, static, h0, contextual = model(
        x, return_attention=True, return_representations=True
    )
    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred = int(probs.argmax())
    tokens = ["[CLS]"] + tokenize(sentence)
    return {
        "tokens": tokens,
        "probs": probs,
        "pred": pred,
        "attn": attn.squeeze(0).cpu(),
        "static": static.squeeze(0).cpu(),
        "h0": h0.squeeze(0).cpu(),
        "contextual": contextual.squeeze(0).cpu(),
    }


def predict_sentence(model, sentence):
    info = model_details(model, sentence)
    pred_label = idx_to_label[info["pred"]]
    print(f"{sentence!r} -> {pred_label} | p(finance)={info['probs'][0]:.2f} p(river)={info['probs'][1]:.2f}")
    return info


def plot_cls_attention(model, sentence, title=None):
    info = model_details(model, sentence)
    tokens = info["tokens"]
    attn = info["attn"].numpy()  # heads, target_tokens, source_tokens
    cls_attention = attn[:, 0, : len(tokens)]

    fig, ax = plt.subplots(figsize=(max(7, len(tokens) * 0.9), 2.2 + 0.35 * cls_attention.shape[0]))
    im = ax.imshow(cls_attention, cmap="YlOrRd", aspect="auto", vmin=0, vmax=max(0.45, float(cls_attention.max())))
    ax.set_xticks(range(len(tokens)), labels=tokens, rotation=30, ha="right")
    ax.set_yticks(range(cls_attention.shape[0]), labels=[f"head {i}" for i in range(cls_attention.shape[0])])
    pred_label = idx_to_label[info["pred"]]
    ax.set_title(title or f"[CLS] attention | prediction: {pred_label}")
    for i in range(cls_attention.shape[0]):
        for j in range(len(tokens)):
            ax.text(j, i, f"{cls_attention[i, j]:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()
    plt.show()


def find_token_index(tokens, token):
    for i, tok in enumerate(tokens):
        if tok == token:
            return i
    raise ValueError(f"{token!r} not found in {tokens}")


def bank_vectors(model, sentence):
    info = model_details(model, sentence)
    bank_idx = find_token_index(info["tokens"], "bank")
    return info, info["static"][bank_idx], info["contextual"][bank_idx]


def compare_bank_contexts(model, sentence_a="bank approves loan", sentence_b="bank has mud near river"):
    info_a, static_a, contextual_a = bank_vectors(model, sentence_a)
    info_b, static_b, contextual_b = bank_vectors(model, sentence_b)
    static_cos = F.cosine_similarity(static_a.unsqueeze(0), static_b.unsqueeze(0)).item()
    contextual_cos = F.cosine_similarity(contextual_a.unsqueeze(0), contextual_b.unsqueeze(0)).item()
    print("Sentence A:", sentence_a)
    print("Sentence B:", sentence_b)
    print(f"Static token-embedding cosine for 'bank':      {static_cos:.3f}")
    print(f"Contextual representation cosine for 'bank': {contextual_cos:.3f}")
    print("\nInterpretation: the token embedding is shared, but self-attention can move the contextual representation apart.")


def plot_contextual_bank_pca(model, examples=None, title="Contextual representations of 'bank'"):
    if examples is None:
        examples = all_examples
    vectors, colors, texts = [], [], []
    for text, y in examples:
        if "bank" not in tokenize(text):
            continue
        info, _, contextual = bank_vectors(model, text)
        vectors.append(contextual.numpy())
        colors.append(y)
        texts.append(text)
    coords = PCA(n_components=2, random_state=0).fit_transform(np.array(vectors))
    plt.figure(figsize=(7, 5))
    palette = {0: "#4c78a8", 1: "#f58518"}
    for label_idx, label in idx_to_label.items():
        mask = np.array(colors) == label_idx
        plt.scatter(coords[mask, 0], coords[mask, 1], label=label, color=palette[label_idx], s=55)
    for i, text in enumerate(texts):
        short = " ".join(tokenize(text)[:3])
        plt.text(coords[i, 0] + 0.03, coords[i, 1] + 0.03, short, fontsize=8)
    plt.title(title)
    plt.xlabel("PCA component 1")
    plt.ylabel("PCA component 2")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Baseline run

Run this first. Do not edit the baseline config.


In [ ]:
baseline_config = {
    "d_model": 32,
    "num_heads": 2,
    "ff_dim": 64,
    "epochs": 120,
    "lr": 0.01,
    "seed": 9,
}

baseline_model, baseline_history = train_transformer(**baseline_config)
plot_training_history(baseline_history, "Baseline tiny transformer")

for sentence in ["bank approves loan", "bank has mud near river"]:
    predict_sentence(baseline_model, sentence)
    plot_cls_attention(baseline_model, sentence)

compare_bank_contexts(baseline_model)
plot_contextual_bank_pca(baseline_model)


### Baseline notes

Write 4-6 sentences:

- What accuracy did the baseline reach?
- Which tokens did `[CLS]` attend to in the two examples?
- What did the contextual `bank` comparison show?
- How is this different from Week 8 attention pooling?


---

## Experiment 1

Change exactly one or two settings compared with the baseline.


In [ ]:
# TODO: change one or two values.
experiment_1_config = {
    "d_model": 24,
    "num_heads": 2,
    "ff_dim": 48,
    "epochs": 100,
    "lr": 0.01,
    "seed": 9,
}

experiment_1_model, experiment_1_history = train_transformer(**experiment_1_config)
plot_training_history(experiment_1_history, "Experiment 1")

for sentence in ["bank manages savings", "river water touches bank"]:
    predict_sentence(experiment_1_model, sentence)
    plot_cls_attention(experiment_1_model, sentence)

compare_bank_contexts(
    experiment_1_model,
    sentence_a="bank manages savings",
    sentence_b="river water touches bank",
)


### Experiment 1 notes

Write 3-5 sentences:

- What did you change?
- What happened to loss or accuracy?
- Did attention behavior change compared with the baseline?


---

## Experiment 2

Try a different controlled change. Keep `d_model` divisible by `num_heads`.


In [ ]:
# TODO: change one or two values.
experiment_2_config = {
    "d_model": 32,
    "num_heads": 4,
    "ff_dim": 64,
    "epochs": 100,
    "lr": 0.01,
    "seed": 9,
}

experiment_2_model, experiment_2_history = train_transformer(**experiment_2_config)
plot_training_history(experiment_2_history, "Experiment 2")

for sentence in ["credit card from bank", "boat rests beside bank"]:
    predict_sentence(experiment_2_model, sentence)
    plot_cls_attention(experiment_2_model, sentence)

compare_bank_contexts(
    experiment_2_model,
    sentence_a="credit card from bank",
    sentence_b="boat rests beside bank",
)


### Experiment 2 notes

Write 3-5 sentences:

- What did you change?
- Which configuration would you keep?
- Does your choice depend only on accuracy, or also on representation behavior?


---

## Custom sentence analysis

Choose at least four custom sentences. Use vocabulary from the fixed dataset when possible.


In [ ]:
# TODO: replace these with your own examples.
my_sentences = [
    "bank transfers cash today",
    "water rises over bank",
    "bank near money",
    "bank near river",
]

chosen_model = baseline_model  # TODO: try experiment_1_model or experiment_2_model

for sentence in my_sentences:
    predict_sentence(chosen_model, sentence)
    plot_cls_attention(chosen_model, sentence)

# TODO: choose two sentences that both contain "bank".
compare_bank_contexts(
    chosen_model,
    sentence_a="bank transfers cash today",
    sentence_b="water rises over bank",
)


### Custom analysis notes

Write 5-8 sentences:

- Which examples were classified confidently?
- Which tokens received high `[CLS]` attention?
- Did attention match your intuition?
- Which contextual `bank` comparison was most convincing?
- Which heatmap looked ambiguous or misleading?
- What is one limitation of this tiny transformer setup?


---

## Final reflection

Answer in 8-12 sentences:

1. What makes self-attention different from Week 8 attention pooling?
2. What did the model learn about `bank` in different contexts?
3. Which configuration worked best and why?
4. What did the attention maps help you inspect?
5. Where should we be careful about interpreting attention?
6. What would change if we used a pretrained transformer instead of this tiny model?


## Academic integrity and AI use

You may use AI assistants for debugging, explanation, and brainstorming.

You remain responsible for understanding the notebook you submit. Be prepared to explain:

- what self-attention does
- why positional embeddings matter
- what `[CLS]` is used for here
- how attention heads were visualized
- how contextual representations differ from static embeddings
